In [ ]:
# =============================================================================
# Acquirung training data for SMOTE testing
# =============================================================================

import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
import joblib
import glob
import os
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings('ignore')

# ── Configuration — must match the main training script ───────────────────────
DATA_DIR = r'C:\Users\lambe\RU_Thesis_2026\ml_data'

DYNAMIC_FEATURES = [
    'temp_max', 'rh_min', 'vpd_mean', 'wind_speed_mean',
    'precip_sum', 'solar_rad_mj', 'rh_7d', 'temp_7d',
    'precip_30d', 'recovery_value'
]
STATIC_FEATURES = [
    'elevation_m', 'slope_deg', 'land_cover',
    'pop_density', 'road_proximity_m'
]
FEATURES       = DYNAMIC_FEATURES + STATIC_FEATURES
TRAIN_YEARS    = list(range(2010, 2021))
VAL_YEARS      = [2021, 2022]
DOY_START_HARD = 91
DOY_END_HARD   = 310

# ── Load training and validation data ─────────────────────────────────────────
def load_years(years, split_label):
    all_dfs = []
    for year in years:
        pattern = os.path.join(DATA_DIR, f'ML_{split_label}_{year}*.csv')
        for f in sorted(glob.glob(pattern)):
            all_dfs.append(pd.read_csv(f))
    if not all_dfs:
        raise FileNotFoundError(f"No CSV files for {split_label} {years}")
    return pd.concat(all_dfs, ignore_index=True)

def clean_and_filter(df):
    required = FEATURES + ['target_ignition', 'fire_cause', 'date']
    df = df.dropna(subset=required)
    df['target_ignition'] = df['target_ignition'].astype(int)
    df['fire_cause']      = df['fire_cause'].astype(int)
    df['land_cover']      = df['land_cover'].astype(int)
    df = df[df['target_ignition'].isin([0, 1])]
    df = df[df['fire_cause'].between(0, 5)]
    df['date_parsed'] = pd.to_datetime(df['date'], format='%Y%m%d')
    df['doy']         = df['date_parsed'].dt.dayofyear
    df = df[(df['doy'] >= DOY_START_HARD) &
            (df['doy'] <= DOY_END_HARD)].copy()
    return df.drop(columns=['date_parsed', 'doy'])

print("Loading training data...")
df_train_raw = load_years(TRAIN_YEARS, 'train')
df_train     = clean_and_filter(df_train_raw)
df_train     = df_train.sort_values('date').reset_index(drop=True)
print(f"  Train rows: {len(df_train):,}")

print("Loading validation data...")
df_val_raw = load_years(VAL_YEARS, 'val')
df_val     = clean_and_filter(df_val_raw)
df_val     = df_val.sort_values('date').reset_index(drop=True)
print(f"  Val rows  : {len(df_val):,}")

# ── Prepare arrays ────────────────────────────────────────────────────────────
X_train     = df_train[FEATURES].values
y_ign_train = df_train['target_ignition'].values
X_val       = df_val[FEATURES].values
y_ign_val   = df_val['target_ignition'].values

fire_tr_mask = df_train['target_ignition'] == 1
n_pos        = fire_tr_mask.sum()
n_neg        = (~fire_tr_mask).sum()
ratio_tr     = n_neg / max(n_pos, 1)

print(f"\nClass balance:")
print(f"  Fire    : {n_pos:,}")
print(f"  No-fire : {n_neg:,}")
print(f"  Ratio   : {ratio_tr:.0f}:1")

# Load CV threshold from previous run
threshold_record = joblib.load('models/xgb_cv_threshold.pkl')
CV_THRESHOLD     = threshold_record['cv_threshold']
print(f"  CV threshold: {CV_THRESHOLD:.4f}")

# ── Now run the tuning subsample block ────────────────────────────────────────
TUNE_FIRE_RATIO = 50

n_fire_tune   = fire_tr_mask.sum()
n_nofire_tune = int(n_fire_tune * TUNE_FIRE_RATIO)

nofire_tune_idx = np.random.RandomState(0).choice(
    np.where(~fire_tr_mask.values)[0], size=n_nofire_tune, replace=False)
tune_idx = np.concatenate([np.where(fire_tr_mask.values)[0],
                            nofire_tune_idx])
np.random.RandomState(0).shuffle(tune_idx)

X_tune = X_train[tune_idx]
y_tune = y_ign_train[tune_idx]

print(f"\nTuning sample: {len(X_tune):,} rows  "
      f"({y_tune.sum():,} fire + {(y_tune==0).sum():,} no-fire)")



In [ ]:
# =============================================================================
# XGBoost Ignition Model — SMOTE Multiplier Experiment
# Tests 10x (baseline), 100x and 1000x SMOTE oversampling
# Uses hyperparameters from model_ignition_final.pkl
# Cross-validates threshold for each model separately
# =============================================================================

import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score,
                             classification_report, confusion_matrix,
                             precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE, SMOTENC
import time
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────────
SMOTE_MULTIPLIERS = [10, 100, 1000]   # fire pixel multipliers to test
MIN_RECALL        = 0.70
N_FOLDS_THRESH    = 5
RANDOM_STATE      = 42

# ── Load hyperparameters from final model ─────────────────────────────────────
print("Loading hyperparameters from model_ignition_final.pkl...")
bundle       = joblib.load('models/model_ignition_final.pkl')
best_params  = bundle['best_params']
CV_THR_ORIG  = bundle['cv_threshold']
FEATURES     = bundle['features']
TRAIN_YEARS  = bundle['train_years']
VAL_YEARS    = bundle['val_years']

print(f"  Loaded best_params  : {best_params}")
print(f"  Original CV threshold: {CV_THR_ORIG:.4f}")
print(f"  Features            : {len(FEATURES)}")

# ── Assumes X_train, y_ign_train, X_val, y_ign_val, ratio_tr in memory ────────
# If not in memory, reload using the memory-efficient loader from the
# main training script before running this block.

print(f"\nTraining data in memory:")
print(f"  X_train     : {X_train.shape}")
print(f"  X_val       : {X_val.shape}")
print(f"  Fire pixels : {y_ign_train.sum():,}")
print(f"  ratio_tr    : {ratio_tr:.0f}:1")

# ── SMOTE helper ──────────────────────────────────────────────────────────────
cat_indices = [FEATURES.index(f) for f in ['land_cover'] if f in FEATURES]

def apply_smote(X_tr, y_tr, multiplier):
    """
    Generate synthetic fire pixels at multiplier × original count.
    Uses SMOTENC if land_cover is present (categorical feature),
    otherwise standard SMOTE.
    """
    n_minority = (y_tr == 1).sum()
    n_target   = n_minority * multiplier
    k          = min(5, n_minority - 1)

    if k < 1:
        print(f"    SMOTE skipped — too few minority samples ({n_minority})")
        return X_tr, y_tr

    print(f"    SMOTE {multiplier}×: {n_minority:,} → {n_target:,} "
          f"fire pixels  (k={k})")

    try:
        sm = (SMOTENC(categorical_features=cat_indices,
                      sampling_strategy={1: n_target},
                      k_neighbors=k,
                      random_state=RANDOM_STATE)
              if cat_indices else
              SMOTE(sampling_strategy={1: n_target},
                    k_neighbors=k,
                    random_state=RANDOM_STATE))
        X_res, y_res = sm.fit_resample(X_tr, y_tr)
        print(f"    Done: {len(X_res):,} total rows  "
              f"({y_res.sum():,} fire + {(y_res==0).sum():,} no-fire)")
        return X_res, y_res
    except Exception as e:
        print(f"    SMOTE failed ({e}) — using original data")
        return X_tr, y_tr

# ── Evaluation helper ─────────────────────────────────────────────────────────
def evaluate_model(model, X, y_true, threshold, label):
    """Print full validation metrics at a given threshold."""
    y_prob = model.predict_proba(X)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    print(f"\n{'='*60}")
    print(f"EVALUATION: {label}")
    print(f"  Threshold         : {threshold:.4f}")
    print(f"{'='*60}")
    print(f"  ROC-AUC           : {roc_auc_score(y_true, y_prob):.4f}")
    print(f"  Balanced accuracy : {balanced_accuracy_score(y_true, y_pred):.4f}")
    print(f"  Recall (fire)     : {tp/(tp+fn):.4f}  "
          f"({tp} of {tp+fn} fires detected)")
    print(f"  Precision (fire)  : {tp/(tp+fp) if (tp+fp)>0 else 0:.4f}")
    print(f"  FPR               : {fp/(fp+tn):.4f}  "
          f"({fp/(fp+tn)*100:.1f}% no-fire days flagged)")
    print(f"  TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}")
    print(classification_report(y_true, y_pred,
                                target_names=['No fire', 'Fire'],
                                digits=4, zero_division=0))
    return {
        'label'    : label,
        'threshold': threshold,
        'auc'      : roc_auc_score(y_true, y_prob),
        'recall'   : tp/(tp+fn),
        'precision': tp/(tp+fp) if (tp+fp) > 0 else 0,
        'fpr'      : fp/(fp+tn),
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn
    }

# ── Threshold CV helper ───────────────────────────────────────────────────────
def compute_cv_threshold(model, X_tr, y_tr, n_folds=N_FOLDS_THRESH,
                         min_recall=MIN_RECALL):
    """
    Compute optimal classification threshold via stratified CV
    on the training set. Maximises precision subject to
    recall >= min_recall.
    """
    skf        = StratifiedKFold(n_splits=n_folds,
                                  shuffle=True,
                                  random_state=RANDOM_STATE)
    candidates = np.linspace(0.01, 0.99, 200)
    fold_thresholds = []

    for fold_idx, (tr_idx, vl_idx) in enumerate(skf.split(X_tr, y_tr)):
        y_prob_f  = model.predict_proba(X_tr[vl_idx])[:, 1]
        best_thr, best_prec = 0.5, 0.0

        for thr in candidates:
            y_t  = (y_prob_f >= thr).astype(int)
            rec  = recall_score(y_tr[vl_idx], y_t, zero_division=0)
            prec = precision_score(y_tr[vl_idx], y_t, zero_division=0)
            if rec >= min_recall and prec > best_prec:
                best_prec, best_thr = prec, thr

        fold_thresholds.append(best_thr)
        print(f"    Fold {fold_idx+1}: threshold={best_thr:.3f}  "
              f"precision={best_prec:.4f}")

    cv_thr     = float(np.mean(fold_thresholds))
    cv_thr_std = float(np.std(fold_thresholds))
    print(f"    CV threshold: {cv_thr:.4f} ± {cv_thr_std:.4f}")
    return cv_thr, cv_thr_std, fold_thresholds

# ── Store all results for comparison ─────────────────────────────────────────
all_results = []
saved_models = {}

# =============================================================================
# MAIN LOOP — train and evaluate one model per SMOTE multiplier
# =============================================================================
for multiplier in SMOTE_MULTIPLIERS:

    print(f"\n{'#'*65}")
    print(f"# SMOTE MULTIPLIER: {multiplier}×")
    print(f"{'#'*65}")

    # ── Step 1: Apply SMOTE to full training set ──────────────────────────────
    print(f"\nStep 1: Applying SMOTE {multiplier}× to training set...")
    t0 = time.time()
    X_smote, y_smote = apply_smote(X_train, y_ign_train, multiplier)
    smote_time = time.time() - t0
    print(f"  SMOTE time: {smote_time/60:.1f} minutes")

    n_fire_smote   = y_smote.sum()
    n_nofire_smote = (y_smote == 0).sum()
    ratio_smote    = n_nofire_smote / max(n_fire_smote, 1)
    print(f"  Augmented ratio: {ratio_smote:.0f}:1  "
          f"({n_fire_smote:,} fire + {n_nofire_smote:,} no-fire)")

    # ── Step 2: Train model with same hyperparameters ─────────────────────────
    print(f"\nStep 2: Training model (SMOTE {multiplier}×)...")
    t0 = time.time()

    model = xgb.XGBClassifier(
        objective             = 'binary:logistic',
        eval_metric           = 'auc',
        scale_pos_weight      = ratio_tr,    # original ratio — not SMOTE ratio
        early_stopping_rounds = 20,
        random_state          = RANDOM_STATE,
        n_jobs                = -1,
        verbosity             = 1,
        **best_params
    )

    model.fit(
        X_smote, y_smote,
        eval_set = [(X_val, y_ign_val)],
        verbose  = 50
    )

    train_time = time.time() - t0
    print(f"  Training time : {train_time/60:.1f} minutes")
    print(f"  Best iteration: {model.best_iteration}")

    # ── Step 3: Evaluate at default threshold 0.5 ────────────────────────────
    r_default = evaluate_model(
        model, X_val, y_ign_val, 0.5,
        f'SMOTE {multiplier}× — default threshold (0.50)')
    all_results.append(r_default)

    # ── Step 4: Cross-validate threshold ─────────────────────────────────────
    print(f"\nStep 3: Cross-validating threshold (SMOTE {multiplier}×)...")
    print(f"  Running {N_FOLDS_THRESH}-fold CV on original training set...")

    cv_thr, cv_thr_std, fold_thrs = compute_cv_threshold(
        model, X_train, y_ign_train)

    # ── Step 5: Evaluate at CV threshold ─────────────────────────────────────
    r_cv = evaluate_model(
        model, X_val, y_ign_val, cv_thr,
        f'SMOTE {multiplier}× — CV threshold ({cv_thr:.3f})')
    all_results.append(r_cv)

    # ── Step 6: Save model ────────────────────────────────────────────────────
    save_path = f'models/model_ignition_smote{multiplier}x.pkl'
    joblib.dump({
        'model'              : model,
        'cv_threshold'       : cv_thr,
        'cv_threshold_std'   : cv_thr_std,
        'fold_thresholds'    : fold_thrs,
        'features'           : FEATURES,
        'best_params'        : best_params,
        'smote_multiplier'   : multiplier,
        'n_fire_original'    : int(y_ign_train.sum()),
        'n_fire_augmented'   : int(n_fire_smote),
        'train_ratio_orig'   : float(ratio_tr),
        'train_ratio_smote'  : float(ratio_smote),
        'smote_time_min'     : smote_time / 60,
        'train_time_min'     : train_time / 60,
        'train_years'        : TRAIN_YEARS,
        'val_years'          : VAL_YEARS,
    }, save_path)
    print(f"\n  Saved: {save_path}")

    saved_models[multiplier] = {
        'model'    : model,
        'cv_thr'   : cv_thr,
        'cv_thr_std': cv_thr_std,
        'path'     : save_path,
        'n_fire_augmented': int(n_fire_smote),
        'ratio_smote'     : float(ratio_smote),
        'train_time'      : train_time,
        'smote_time'      : smote_time,
    }

# =============================================================================
# FINAL COMPARISON TABLE — all multipliers at both thresholds
# =============================================================================
print(f"\n{'='*75}")
print("SMOTE MULTIPLIER EXPERIMENT — SUMMARY")
print(f"{'='*75}")
print(f"  Hyperparameters : from model_ignition_final.pkl")
print(f"  Original threshold (10×): {CV_THR_ORIG:.4f}")
print(f"\n  {'Stage':<50} {'AUC':>7} {'Recall':>8} "
      f"{'FPR':>8} {'TP':>7} {'FP':>10}")
print(f"  {'-'*92}")

for r in all_results:
    print(f"  {r['label']:<50} {r['auc']:>7.4f} "
          f"{r['recall']:>8.4f} {r['fpr']:>8.4f} "
          f"{r['tp']:>7,} {r['fp']:>10,}")

# ── Clean comparison at CV threshold only ─────────────────────────────────────
print(f"\n{'='*75}")
print("CV THRESHOLD RESULTS ONLY — CLEAN COMPARISON")
print(f"{'='*75}")
print(f"  {'Multiplier':<15} {'CV Thr':>8} {'AUC':>7} {'Recall':>8} "
      f"{'Precision':>11} {'FPR':>8} {'TP':>7} {'FP':>10}")
print(f"  {'-'*80}")

for multiplier, info in saved_models.items():
    m   = info['model']
    thr = info['cv_thr']
    y_prob = m.predict_proba(X_val)[:, 1]
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_ign_val, y_pred).ravel()
    auc  = roc_auc_score(y_ign_val, y_prob)
    rec  = tp / (tp + fn)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    fpr  = fp / (fp + tn)
    print(f"  {multiplier}×{'':<12} {thr:>8.4f} {auc:>7.4f} {rec:>8.4f} "
          f"{prec:>11.4f} {fpr:>8.4f} {tp:>7,} {fp:>10,}")

# ── Training stats ─────────────────────────────────────────────────────────────
print(f"\n{'='*75}")
print("TRAINING STATISTICS PER MULTIPLIER")
print(f"{'='*75}")
print(f"  {'Multiplier':<15} {'Fire orig':>10} {'Fire aug':>10} "
      f"{'Ratio':>8} {'SMOTE (min)':>12} {'Train (min)':>12}")
print(f"  {'-'*70}")
n_fire_orig = int(y_ign_train.sum())
for multiplier, info in saved_models.items():
    print(f"  {multiplier}×{'':<12} {n_fire_orig:>10,} "
          f"{info['n_fire_augmented']:>10,} "
          f"{info['ratio_smote']:>7.0f}:1 "
          f"{info['smote_time']/60:>12.1f} "
          f"{info['train_time']/60:>12.1f}")

print(f"\nSaved models:")
for multiplier, info in saved_models.items():
    print(f"  {info['path']}")
    print(f"    CV threshold : {info['cv_thr']:.4f} ± "
          f"{info['cv_thr_std']:.4f}")